# 🎀 林星蕾 LoRA 訓練營 v3 — 全自動版
## 設備：T4 GPU · SDXL · 1024×1024 · 118張圖片
---
**全自動：下載素材 → 訓練 → 下載 LoRA**
---
### 訓練參數
| 項目 | 數值 |
|------|------|
| 基底模型 | SDXL 1.0 |
| 解析度 | 1024×1024 |
| LoRA dim/alpha | 32 / 16 |
| 訓練步數 | 1200 (~10 epochs) |
| 學習率 | 1e-4 (cosine) |
| 觸發詞 | `linxinglei` |
| 優化器 | AdamW8bit |

In [ ]:
# === 1. 安裝依賴（不使用 kohya_ss requirements，避免版本衝突）===
!pip install -q --upgrade pip
!pip install -q accelerate transformers ftfy tensorboard safetensors lion-pytorch huggingface_hub opencv-python-headless
print('✅ 依賴安裝完成')

In [ ]:
# === 2. 設定 Accelerate ===
from accelerate.utils import write_basic_config
write_basic_config(mixed_precision='bf16')
print('✅ Accelerate 設定完成 (bf16)')

In [ ]:
# === 3. 下載 lora_training_data.zip ===
import urllib.request, os

ZIP_URL = 'https://github.com/vito0527opencode/lora-training/raw/main/lora_training_data.zip'
ZIP_PATH = '/content/lora_training_data.zip'

if not os.path.exists(ZIP_PATH):
    print('📦 自動下載訓練素材...')
    urllib.request.urlretrieve(ZIP_URL, ZIP_PATH)
    print(f'✅ 下載完成: {os.path.getsize(ZIP_PATH)//1024//1024} MB')
else:
    print(f'✅ zip 已存在')

assert os.path.exists(ZIP_PATH), 'zip 下載失敗！'
print(f'📦 zip 大小: {os.path.getsize(ZIP_PATH)//1024//1024} MB')

In [ ]:
# === 4. 解壓縮 + 產生 caption 檔案 ===
import zipfile, glob, os, random, re

zips = glob.glob('/content/*.zip')
if zips:
    with zipfile.ZipFile(zips[0], 'r') as z:
        z.extractall('/content/training_data')
    print('✅ 解壓縮完成')

# 找 caption 檔位置（兩個可能：根目錄或 processed/ 子目錄）
caption_candidates = [
    '/content/training_data/captions.txt',
    '/content/training_data/processed/captions.txt'
]
caption_file = None
for c in caption_candidates:
    if os.path.exists(c):
        caption_file = c
        break

assert caption_file, f'找不到 captions.txt！檢查過: {caption_candidates}'
print(f'✅ Caption 檔: {caption_file}')

# 找到 processed 目錄
processed_dir = os.path.dirname(caption_file)
print(f'✅ 訓練資料目錄: {processed_dir}')

# 讀 caption 對應表（格式：檔名|caption）
TRIGGER = 'linxinglei'
caption_map = {}
with open(caption_file, 'r', encoding='utf-8') as f:
    for line in f:
        line = line.strip()
        if '|' not in line:
            continue
        img_name, cap = line.split('|', 1)
        caption_map[img_name.strip()] = cap.strip()

# 為所有 PNG 產生 .txt caption 檔
png_files = [f for f in os.listdir(processed_dir) if f.endswith('.png')]
txt_count = 0
for png in png_files:
    txt_path = os.path.join(processed_dir, png.rsplit('.', 1)[0] + '.txt')
    # caption 從 map 取，找不到就用預設值
    cap = caption_map.get(png, f'a woman with long black hair')
    with open(txt_path, 'w', encoding='utf-8') as cf:
        cf.write(f'{TRIGGER}, {cap}')
    txt_count += 1

print(f'✅ 產生 {txt_count} 個 caption 檔案（觸發詞: {TRIGGER}）')

# 抽樣顯示
samples = random.sample(png_files, min(5, len(png_files)))
for s in sorted(samples):
    txt_path = os.path.join(processed_dir, s.rsplit('.', 1)[0] + '.txt')
    cap = open(txt_path).read().strip() if os.path.exists(txt_path) else '(無)'
    print(f'  📸 {s} → {cap}')

In [ ]:
# === 5. 直接克隆 Kohya sd-scripts（不裝 kohya_ss GUI 相關依賴）===
!git clone -q https://github.com/kohya-ss/sd-scripts.git /content/sd-scripts
%cd /content/sd-scripts

import os
TRAIN_SCRIPT = '/content/sd-scripts/sdxl_train_network.py'
assert os.path.exists(TRAIN_SCRIPT), f'❌ 找不到腳本: {TRAIN_SCRIPT}'

print(f'✅ sd-scripts 安裝完成')
print(f'📝 訓練腳本: {TRAIN_SCRIPT}')


In [ ]:
# === 6. 🚀 開始訓練（SDXL LoRA）===
import os, subprocess, sys

# 確認 processed 目錄
processed_dir = None
for root, dirs, files in os.walk('/content/training_data'):
    if 'captions.txt' in files:
        processed_dir = root
        break
if not processed_dir:
    for cand in ['/content/training_data/processed', '/content/training_data']:
        if os.path.exists(cand):
            processed_dir = cand
            break

TRAIN_SCRIPT = '/content/sd-scripts/sdxl_train_network.py'
assert os.path.exists(TRAIN_SCRIPT), f'❌ 找不到腳本: {TRAIN_SCRIPT}'

print(f'📂 訓練資料: {processed_dir}')
print(f'📂 輸出目錄: /content/output')
print(f'📝 腳本路徑: {TRAIN_SCRIPT}')
print()

# GPU 檢查
!nvidia-smi
print()
print('🔥 開始訓練... (約 1-2 小時)')
print('='*50)

os.makedirs('/content/output', exist_ok=True)

# 組裝完整指令（SDXL 專用）
cmd = (
    f'accelerate launch --num_cpu_threads_per_process 4 '
    f'{TRAIN_SCRIPT} '
    f'--pretrained_model_name_or_path=stabilityai/stable-diffusion-xl-base-1.0 '
    f'--train_data_dir={processed_dir} '
    f'--output_dir=/content/output '
    f'--output_name=linxinglei_lora_v1 '
    f'--resolution=1024,1024 '
    f'--train_batch_size=1 '
    f'--network_module=networks.lora '
    f'--network_dim=32 '
    f'--network_alpha=16 '
    f'--max_train_steps=1200 '
    f'--learning_rate=1e-4 '
    f'--optimizer_type=AdamW8bit '
    f'--mixed_precision=bf16 '
    f'--save_every_n_epochs=2 '
    f'--save_model_as=safetensors '
    f'--caption_extension=.txt '
    f'--cache_latents '
    f'--lr_scheduler=cosine '
    f'--lr_warmup_steps=50 '
    f'--enable_bucket '
    f'--min_bucket_reso=256 '
    f'--max_bucket_reso=2048 '
    f'--keep_tokens=0 '
    f'--no_half_vae '
    f'--gradient_accumulation_steps=1'
)

# 執行訓練
result = subprocess.run(cmd, shell=True)
if result.returncode != 0:
    print()
    print('❌ 訓練失敗！')
    print('請查看上方錯誤訊息')
    sys.exit(1)

print()
print('='*50)
print('🎉 訓練完成！')
print('='*50)


In [ ]:
# === 7. 下載 LoRA ===
import glob, os
from datetime import datetime

print('📦 尋找 LoRA 檔案...')

safetensors_files = glob.glob('/content/output/**/*.safetensors', recursive=True)

if not safetensors_files:
    for alt in ['/content/sd-scripts/output', '/content/drive/MyDrive']:
        safetensors_files = glob.glob(f'{alt}/**/*.safetensors', recursive=True)
        if safetensors_files:
            break

if safetensors_files:
    latest = max(safetensors_files, key=os.path.getmtime)
    size_mb = os.path.getsize(latest) / 1024 / 1024
    mtime = datetime.fromtimestamp(os.path.getmtime(latest))
    print(f'📦 找到 LoRA: {latest}')
    print(f'📏 大小: {size_mb:.1f} MB')
    print(f'🕐 時間: {mtime}')
    print()
    print('⬇️  開始下載...')
    from google.colab import files
    files.download(latest)
else:
    print('⚠️ 找不到 .safetensors 檔案')
    print('檢查 /content/output/ 目錄：')
    !find /content/output -type f 2>/dev/null | head -20

In [ ]:
# === 8. 備份到 Google Drive ===
from google.colab import drive
import shutil, os, glob

try:
    drive.mount('/content/drive')
    print('✅ Google Drive 已掛載')
    
    output = '/content/output'
    backup_dir = '/content/drive/MyDrive/LoRA/linxinglei'
    os.makedirs(backup_dir, exist_ok=True)
    
    safetensors_files = glob.glob(f'{output}/**/*.safetensors', recursive=True)
    for f in safetensors_files:
        dest = os.path.join(backup_dir, os.path.basename(f))
        shutil.copy2(f, dest)
        print(f'📁 已備份: {dest}')
    print('✅ 備份完成')
except Exception as e:
    print(f'ℹ️ 跳過 Drive 備份: {e}')

---
## 🎉 完成！

LoRA 已下載到電腦。

### 使用方式
```
在 ComfyUI / A1111 / SD.Next 載入 linxinglei_lora_v1.safetensors
觸發詞: linxinglei
權重建議: 0.6 ~ 0.8
```